In [15]:
from pathlib import Path
import pandas as pd
import duckdb

SOURCE_FILE = Path("../data/raw/amazon.csv")
BRONZE_FILE = "../data/bronze/amazon_bronze.parquet"
db_path = Path("../warehouse/amazon.duckdb")

con = duckdb.connect(db_path, read_only=False)


In [32]:
con.execute(
    """
COPY (SELECT * exclude (discounted_price, actual_price, discount_percentage, rating, rating_count),
TRY_CAST(REPLACE(REPLACE(discounted_price, ',', ''), '₹', '') AS DOUBLE) AS discounted_price,
TRY_CAST(REPLACE(REPLACE(actual_price, ',', ''), '₹', '') AS DOUBLE) AS actual_price,
TRY_CAST(REPLACE(discount_percentage, '%', '') AS DOUBLE) AS discount_percentage,
TRY_CAST(REPLACE(rating, '|', '') AS DOUBLE) AS rating,
TRY_CAST(REPLACE(rating_count, ',', '') AS BIGINT) AS rating_count
FROM read_csv_auto(?)
) TO ? (FORMAT PARQUET);
""",
    [str(SOURCE_FILE), str(BRONZE_FILE)],
)

print(f"Bronze data written to {BRONZE_FILE}")

Bronze data written to ../data/bronze/amazon_bronze.parquet
